# EMCAD ClinicDB Replication — Split Training Run 1



The restored author repository is kept unchanged. A temporary execution copy is created, and the **only change** in that copy is the outer run selector:

`for run in [1,2,3,4,5]:` → `for run in [1]:`

No architecture, loss, optimizer, learning rate, scheduler, augmentation, dataset split, batch size, image size, epoch count, validation logic, or checkpoint-selection logic is changed.

In [1]:
RUN_NUMBER=1
assert RUN_NUMBER in [1,2,3,4,5]
print("Independent run:",RUN_NUMBER)

Independent run: 1


In [2]:
!nvidia-smi
!python --version

Mon Sep 28 05:05:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path

input_root=Path("/kaggle/input")

env_archives=list(input_root.rglob("emcadenv.tar.gz"))
repo_archives=list(input_root.rglob("EMCAD_prepared.tar.gz"))
checksum_files=list(input_root.rglob("EMCAD_NOTEBOOK1_SHA256SUMS.txt"))

assert len(env_archives)==1,f"Expected one emcadenv.tar.gz, found {env_archives}"
assert len(repo_archives)==1,f"Expected one EMCAD_prepared.tar.gz, found {repo_archives}"
assert len(checksum_files)==1,f"Expected one checksum file, found {checksum_files}"

notebook1_root=env_archives[0].parent
assert repo_archives[0].parent==notebook1_root
assert checksum_files[0].parent==notebook1_root

raw_pairs=[]
for image_root in input_root.rglob("Original"):
    if not image_root.is_dir():
        continue
    mask_root=image_root.parent/"Ground Truth"
    if not mask_root.is_dir():
        continue
    images=[p for p in image_root.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}]
    masks=[p for p in mask_root.iterdir() if p.is_file() and p.suffix.lower() in {".tif",".tiff"}]
    if len(images)==612 and len(masks)==612:
        raw_pairs.append((image_root,mask_root))

assert len(raw_pairs)==1,f"Expected one raw ClinicDB input, found {len(raw_pairs)}"

print("Notebook 1 output:",notebook1_root)
print("Environment archive:",env_archives[0])
print("EMCAD archive:",repo_archives[0])
print("Raw ClinicDB images:",raw_pairs[0][0])
print("Raw ClinicDB masks:",raw_pairs[0][1])
print("Raw counts: 612 images, 612 masks")

Notebook 1 output: /kaggle/input/notebooks/anirbanroysudipto/notebook-1-environment-preparation
Environment archive: /kaggle/input/notebooks/anirbanroysudipto/notebook-1-environment-preparation/emcadenv.tar.gz
EMCAD archive: /kaggle/input/notebooks/anirbanroysudipto/notebook-1-environment-preparation/EMCAD_prepared.tar.gz
Raw ClinicDB images: /kaggle/input/datasets/anirbanroysudipto/cvc-clinicdb-raw/CVC-ClinicDB/Original
Raw ClinicDB masks: /kaggle/input/datasets/anirbanroysudipto/cvc-clinicdb-raw/CVC-ClinicDB/Ground Truth
Raw counts: 612 images, 612 masks


In [4]:
%%bash
set -euo pipefail

ROOT=$(dirname "$(find /kaggle/input -name EMCAD_NOTEBOOK1_SHA256SUMS.txt -print -quit)")
cd "$ROOT"

sha256sum -c EMCAD_NOTEBOOK1_SHA256SUMS.txt

rm -rf /kaggle/working/emcadenv
rm -rf /kaggle/working/EMCAD_AUTHOR
rm -rf /kaggle/working/EMCAD_EXEC

mkdir -p /kaggle/working/emcadenv
tar -xzf emcadenv.tar.gz -C /kaggle/working/emcadenv
/kaggle/working/emcadenv/bin/conda-unpack

mkdir -p /kaggle/working/restore_tmp
rm -rf /kaggle/working/restore_tmp/*
tar -xzf EMCAD_prepared.tar.gz -C /kaggle/working/restore_tmp
mv /kaggle/working/restore_tmp/EMCAD /kaggle/working/EMCAD_AUTHOR
rm -rf /kaggle/working/restore_tmp

cp -a /kaggle/working/EMCAD_AUTHOR /kaggle/working/EMCAD_EXEC

echo "Notebook 1 environment and repository restored."


emcadenv.tar.gz: OK
EMCAD_prepared.tar.gz: OK
EMCAD_AUTHOR_COMMIT.txt: OK
ClinicDB_author_split_manifest.csv: OK
EMCAD_ENVIRONMENT.txt: OK
PVT_V2_B2_SHA256.txt: OK
Notebook 1 environment and repository restored.


In [5]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
REPO=/kaggle/working/EMCAD_AUTHOR
EXPECTED=26c9c31f731f749b62c5fe83f44376dac75f3aa8

cd "$REPO"

test "$(git rev-parse HEAD)" = "$EXPECTED"
git diff --exit-code
git diff --cached --exit-code
git status --porcelain --untracked-files=no

$ENV/bin/python - <<'PY'
import sys
import torch
import numpy
import timm
import ptflops

print("Python:",sys.version.split()[0])
print("PyTorch:",torch.__version__)
print("CUDA build:",torch.version.cuda)
print("CUDA available:",torch.cuda.is_available())
print("GPU count:",torch.cuda.device_count())
print("GPU 0:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("NumPy:",numpy.__version__)
print("timm:",timm.__version__)
print("ptflops:",ptflops.__version__ if hasattr(ptflops,"__version__") else "0.7.2.2")

assert sys.version_info[:2]==(3,8)
assert torch.__version__.startswith("1.11.0")
assert torch.cuda.is_available()
PY

echo "Original author repository and environment verified."


Python: 3.8.20
PyTorch: 1.11.0+cu113
CUDA build: 11.3
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
NumPy: 1.22.4
timm: 0.6.12
ptflops: 0.7.2.2
Original author repository and environment verified.


In [6]:
%%bash
set -euo pipefail

REPO=/kaggle/working/EMCAD_AUTHOR
cd "$REPO"

for split in train val test; do
    ni=$(find "data/polyp/target/ClinicDB/$split/images" -maxdepth 1 -type f -name "*.png" | wc -l)
    nm=$(find "data/polyp/target/ClinicDB/$split/masks" -maxdepth 1 -type f -name "*.png" | wc -l)
    echo "$split images=$ni masks=$nm"
done

test "$(find data/polyp/target/ClinicDB/train/images -maxdepth 1 -type f -name "*.png" | wc -l)" -eq 489
test "$(find data/polyp/target/ClinicDB/val/images -maxdepth 1 -type f -name "*.png" | wc -l)" -eq 61
test "$(find data/polyp/target/ClinicDB/test/images -maxdepth 1 -type f -name "*.png" | wc -l)" -eq 62

echo "Prepared ClinicDB split verified."


train images=489 masks=489
val images=61 masks=61
test images=62 masks=62
Prepared ClinicDB split verified.


## Create the one-run execution copy

The next cell changes exactly one line in the temporary execution copy. It also writes a unified diff so the change is fully auditable.

In [7]:
from pathlib import Path
import difflib
import hashlib
import re

run_number=int(RUN_NUMBER)

author_file=Path("/kaggle/working/EMCAD_AUTHOR/train_polyp.py")
exec_file=Path("/kaggle/working/EMCAD_EXEC/train_polyp.py")
patch_file=Path(f"/kaggle/working/RUN_{run_number}_SELECTOR.patch")

original=author_file.read_text()
execution=exec_file.read_text()

assert original==execution

pattern=r"for run in \[1\s*,\s*2\s*,\s*3\s*,\s*4\s*,\s*5\]:"
replacement=f"for run in [{run_number}]:"

modified,count=re.subn(pattern,replacement,execution,count=1)
assert count==1,"Could not identify the authors' five-run loop exactly."

exec_file.write_text(modified)

diff="".join(difflib.unified_diff(
    original.splitlines(keepends=True),
    modified.splitlines(keepends=True),
    fromfile="AUTHOR/train_polyp.py",
    tofile=f"EXECUTION_RUN_{run_number}/train_polyp.py"
))

changed=[
    line for line in diff.splitlines()
    if (line.startswith("+") or line.startswith("-"))
    and not line.startswith("+++")
    and not line.startswith("---")
]

assert len(changed)==2,changed
assert any(replacement in line for line in changed)

patch_file.write_text(diff)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

print(diff)
print("Original SHA256:",sha256(author_file))
print("Execution SHA256:",sha256(exec_file))
print("Only the outer run selector was changed.")


--- AUTHOR/train_polyp.py
+++ EXECUTION_RUN_1/train_polyp.py
@@ -211,7 +211,7 @@
     parser.add_argument('--train_save', type=str, default='') 
     opt = parser.parse_args()
 
-    for run in [1,2,3,4,5]:
+    for run in [1]:
         dict_plot = {'val': [], 'test': []}
         best = 0.0
         test_dice_at_best_val = 0.0

Original SHA256: af7a95a7b92eb3f0622cfb317b89e2c0dd5157e3f066e18a5a4fa0e95f4c0abc
Execution SHA256: 4b799e86a8efde281f0805c94b8365588253496985078390a16f15fd85c802d5
Only the outer run selector was changed.


In [8]:
%%bash
set -euo pipefail

AUTHOR=/kaggle/working/EMCAD_AUTHOR
EXEC=/kaggle/working/EMCAD_EXEC

cd "$AUTHOR"
git diff --exit-code
git diff --cached --exit-code
git status --porcelain --untracked-files=no

echo "Original author repository remains unchanged."

cd "$EXEC"
git diff -- train_polyp.py


Original author repository remains unchanged.
diff --git a/train_polyp.py b/train_polyp.py
index 9630b2b..d0875a1 100644
--- a/train_polyp.py
+++ b/train_polyp.py
@@ -211,7 +211,7 @@ if __name__ == '__main__':
     parser.add_argument('--train_save', type=str, default='') 
     opt = parser.parse_args()
 
-    for run in [1,2,3,4,5]:
+    for run in [1]:
         dict_plot = {'val': [], 'test': []}
         best = 0.0
         test_dice_at_best_val = 0.0


## Official training logic — Run 1

The next cell executes the authors' training script from the temporary one-run execution copy. All original training settings remain unchanged, including 200 epochs.

In [9]:
%%bash
set -euo pipefail

ENV=/kaggle/working/emcadenv
EXEC=/kaggle/working/EMCAD_EXEC

cd "$EXEC"

rm -rf model_pth logs results_polyp predictions_polyp
rm -f All_Runs_Summary_Polyp.xlsx
rm -f /kaggle/working/author_train_stdout.log

START=$(date +%s)

$ENV/bin/python -u -W ignore train_polyp.py 2>&1 | tee /kaggle/working/author_train_stdout.log

END=$(date +%s)
echo $((END-START)) > /kaggle/working/wall_clock_seconds.txt

echo "Training process finished."


Model pvt_v2_b2 backbone:  created, param count: 24849856
Model EMCAD decoder:  created, param count: 1913515
Encoder: pvt_v2_b2 | Decoder: EMCAD
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.batchnorm.BatchNorm2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_normalization() for <class 'torch.nn.modules.normalization.LayerNorm'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.dropout.Dropout'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU6'>.
[INFO] Register count_upsample() for <class 'torch.nn.modules.upsampling.Upsample'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for

In [10]:
from pathlib import Path
import re

run_number=int(RUN_NUMBER)
repo=Path("/kaggle/working/EMCAD_EXEC")
run_dirs=sorted([p for p in (repo/"model_pth").iterdir() if p.is_dir()])

assert len(run_dirs)==1,f"Expected exactly one run directory, found {len(run_dirs)}"

run_dir=run_dirs[0]
run_id=run_dir.name

assert f"_run{run_number}_" in run_id,run_id
assert (run_dir/f"{run_id}-best.pth").is_file()
assert (run_dir/f"{run_id}-last.pth").is_file()

log_path=repo/"logs"/f"train_log_{run_id}.log"
assert log_path.is_file()

stdout=Path("/kaggle/working/author_train_stdout.log").read_text(errors="ignore")

assert "Epoch [200/200]" in stdout
assert "FINAL RESULTS:" in stdout

m_best=re.findall(r"Best Val Dice:\s*([0-9.]+)",stdout)
m_test=re.findall(r"Test Dice at Best Val:\s*([0-9.]+)",stdout)
m_time=re.findall(r"Total Train Time:\s*([0-9.]+)s",stdout)

assert m_best and m_test and m_time

print("Run number:",run_number)
print("Run ID:",run_id)
print("Best Val Dice:",m_best[-1])
print("Test Dice at Best Val:",m_test[-1])
print("Author Total Train Time:",m_time[-1],"s")
print("Best checkpoint:",run_dir/f"{run_id}-best.pth")
print("Last checkpoint:",run_dir/f"{run_id}-last.pth")


Run number: 1
Run ID: ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008
Best Val Dice: 0.9217
Test Dice at Best Val: 0.9401
Author Total Train Time: 37533.87 s
Best checkpoint: /kaggle/working/EMCAD_EXEC/model_pth/ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008/ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008-best.pth
Last checkpoint: /kaggle/working/EMCAD_EXEC/model_pth/ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008/ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008-last.pth


## Package this independent run

No testing or post-hoc analysis is performed here. Those steps will be done after Runs 1–5 have all completed.

In [11]:
from pathlib import Path
import json
import hashlib
import re
import shutil
import subprocess

run_number=int(RUN_NUMBER)
author=Path("/kaggle/working/EMCAD_AUTHOR")
execution=Path("/kaggle/working/EMCAD_EXEC")
working=Path("/kaggle/working")

run_dirs=sorted([p for p in (execution/"model_pth").iterdir() if p.is_dir()])
assert len(run_dirs)==1
run_dir=run_dirs[0]
run_id=run_dir.name

package=working/f"EMCAD_run{run_number}_training_output"
if package.exists():
    shutil.rmtree(package)

(package/"model_pth"/run_id).mkdir(parents=True)
(package/"logs").mkdir(parents=True)
(package/"evidence").mkdir(parents=True)

shutil.copy2(run_dir/f"{run_id}-best.pth",package/"model_pth"/run_id/f"{run_id}-best.pth")
shutil.copy2(run_dir/f"{run_id}-last.pth",package/"model_pth"/run_id/f"{run_id}-last.pth")
shutil.copy2(execution/"logs"/f"train_log_{run_id}.log",package/"logs"/f"train_log_{run_id}.log")
shutil.copy2(working/"author_train_stdout.log",package/"author_train_stdout.log")
shutil.copy2(working/f"RUN_{run_number}_SELECTOR.patch",package/"evidence"/f"RUN_{run_number}_SELECTOR.patch")
shutil.copy2(author/"train_polyp.py",package/"evidence"/"author_train_polyp.py")
shutil.copy2(execution/"train_polyp.py",package/"evidence"/f"execution_run{run_number}_train_polyp.py")

commit=subprocess.check_output(["git","-C",str(author),"rev-parse","HEAD"],text=True).strip()
status=subprocess.check_output(
    ["git","-C",str(author),"status","--porcelain","--untracked-files=no"],
    text=True
).strip()

stdout=(working/"author_train_stdout.log").read_text(errors="ignore")

def last(pattern):
    m=re.findall(pattern,stdout)
    return m[-1] if m else None

metadata={
    "run_number":run_number,
    "run_id":run_id,
    "author_commit":commit,
    "author_tracked_status":status,
    "dataset":"ClinicDB",
    "train_count":489,
    "val_count":61,
    "test_count":62,
    "encoder":"pvt_v2_b2",
    "decoder":"EMCAD",
    "kernel_sizes":[1,3,5],
    "lgag_kernel_size":3,
    "expansion_factor":2,
    "image_size":352,
    "batch_size":8,
    "epochs":200,
    "learning_rate":0.0005,
    "best_val_dice":float(last(r"Best Val Dice:\s*([0-9.]+)")),
    "test_dice_at_best_val":float(last(r"Test Dice at Best Val:\s*([0-9.]+)")),
    "author_total_train_time_seconds":float(last(r"Total Train Time:\s*([0-9.]+)s")),
    "wall_clock_seconds":int((working/"wall_clock_seconds.txt").read_text().strip()),
    "execution_difference":"Only the outer run selector [1,2,3,4,5] was restricted to this independent run."
}

(package/"run_metadata.json").write_text(json.dumps(metadata,indent=2))

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

files=[
    package/"model_pth"/run_id/f"{run_id}-best.pth",
    package/"model_pth"/run_id/f"{run_id}-last.pth",
    package/"logs"/f"train_log_{run_id}.log",
    package/"author_train_stdout.log",
    package/"evidence"/f"RUN_{run_number}_SELECTOR.patch",
    package/"evidence"/"author_train_polyp.py",
    package/"evidence"/f"execution_run{run_number}_train_polyp.py",
    package/"run_metadata.json"
]

with (package/"SHA256SUMS.txt").open("w") as f:
    for path in files:
        f.write(f"{sha256(path)}  {path.relative_to(package)}\n")

archive=shutil.make_archive(
    str(working/f"EMCAD_run{run_number}_training_output"),
    "gztar",
    root_dir=working,
    base_dir=package.name
)

print(json.dumps(metadata,indent=2))
print()
print("Archive:",archive)
print("Size:",f"{Path(archive).stat().st_size/1024**2:.2f} MB")


{
  "run_number": 1,
  "run_id": "ClinicDB_pvt_v2_b2_EMCAD_kernel_sizes_1_3_5_dw_parallel_add_lgag_ks_3_ef2_act_mscb_relu6_bs8_cas_lr0.0005_e200_augTrue_run1_t051008",
  "author_commit": "26c9c31f731f749b62c5fe83f44376dac75f3aa8",
  "author_tracked_status": "",
  "dataset": "ClinicDB",
  "train_count": 489,
  "val_count": 61,
  "test_count": 62,
  "encoder": "pvt_v2_b2",
  "decoder": "EMCAD",
  "kernel_sizes": [
    1,
    3,
    5
  ],
  "lgag_kernel_size": 3,
  "expansion_factor": 2,
  "image_size": 352,
  "batch_size": 8,
  "epochs": 200,
  "learning_rate": 0.0005,
  "best_val_dice": 0.9217,
  "test_dice_at_best_val": 0.9401,
  "author_total_train_time_seconds": 37533.87,
  "wall_clock_seconds": 38260,
  "execution_difference": "Only the outer run selector [1,2,3,4,5] was restricted to this independent run."
}

Archive: /kaggle/working/EMCAD_run1_training_output.tar.gz
Size: 189.73 MB


In [12]:
%%bash
set -euo pipefail

AUTHOR=/kaggle/working/EMCAD_AUTHOR

cd "$AUTHOR"
git diff --exit-code
git diff --cached --exit-code
git status --porcelain --untracked-files=no

echo "Final check: original author repository is still unchanged."

ls -lh /kaggle/working/EMCAD_run*_training_output.tar.gz


Final check: original author repository is still unchanged.
-rw-r--r-- 1 root root 190M Sep 28 15:47 /kaggle/working/EMCAD_run1_training_output.tar.gz


In [13]:
from pathlib import Path
import json

run_number=int(RUN_NUMBER)
package=Path(f"/kaggle/working/EMCAD_run{run_number}_training_output")
metadata=json.loads((package/"run_metadata.json").read_text())

print("RUN COMPLETE")
print("Run number:",metadata["run_number"])
print("Best Val Dice:",metadata["best_val_dice"])
print("Test Dice at Best Val:",metadata["test_dice_at_best_val"])
print("Training time:",metadata["author_total_train_time_seconds"],"s")
print()
print("Save this Kaggle notebook version with outputs.")
print(f"Required output: EMCAD_run{run_number}_training_output.tar.gz")


RUN COMPLETE
Run number: 1
Best Val Dice: 0.9217
Test Dice at Best Val: 0.9401
Training time: 37533.87 s

Save this Kaggle notebook version with outputs.
Required output: EMCAD_run1_training_output.tar.gz
